# Seasonal hotspots in satellite composites

This notebook takes a stack of gridded composites (one CSV per acquisition, lattice of
longitude, latitude, value), bins them to a hexagonal grid, builds seasonal covariates per hex,
and screens each covariate for spatial clusters with Getis-Ord Gi* and local Moran's I. The
outputs are a hex GeoPackage with hotspot classes, QML styles, and a Kepler.gl map with one layer
per covariate.

The composites are synthetic chlorophyll-a fields: four years, six days of year each, on a 60 by
60 degree lattice over the North Atlantic, with a seasonal cycle, a coastal gradient, a slow
interannual trend and 8 percent cloud gaps. The reader only needs the acquisition date in the
filename (`AYYYYDDD` by default), so real composites drop in without code changes.

Dask is used for the point to hex join. The cluster dashboard link is printed when the client
starts; on WSL the `localhost` form opens in the Windows browser. `DASK_N_WORKERS`,
`DASK_THREADS_PER_WORKER`, `DASK_MEMORY_LIMIT` and `DASK_DASHBOARD_ADDRESS` override the
defaults; `DASK_OPEN_DASHBOARD=1` opens the page as the client starts.

In [1]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
warnings.filterwarnings("ignore")

DATA = ROOT / "data" / "oceanography"
EXPORTS = ROOT / "exports" / "oceanography"
DATA.mkdir(parents=True, exist_ok=True)
EXPORTS.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
print("repo root:", ROOT)

repo root: /mnt/c/Users/joogl/OneDrive/Documents/VisualStudioCodeProjects/Fortress-GIS-Projects


In [2]:
from datasets.synthetic import synthetic_chlorophyll_grids
from fortress_gis.compute.cluster import get_dask_client
from fortress_gis.domains import oceanography as oc
from fortress_gis.features.seasonal import missingness_report
from fortress_gis.stats.hypothesis import benjamini_hochberg
from fortress_gis.viz.kepler import KeplerMapBuilder, kepler_available

paths = sorted(DATA.glob("A*_chlor_a.csv"))
if not paths:
    paths = synthetic_chlorophyll_grids(
        DATA,
        years=(2016, 2017, 2018, 2019),
        days_of_year=(15, 60, 135, 200, 250, 320),
        lon=(-40.0, -10.0, 60),
        lat=(25.0, 55.0, 60),
    )
print(len(paths), "acquisitions, first:", paths[0].name)

24 acquisitions, first: A2016015_chlor_a.csv


In [3]:
client = get_dask_client()  # prints the dashboard link; DASK_N_WORKERS etc. override defaults
client

Dask dashboard (Windows browser): http://localhost:8787/status
Dask dashboard (WSL): http://127.0.0.1:8787/status


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 3
Total threads: 3,Total memory: 18.16 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42885,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:34117,Total threads: 1
Dashboard: http://127.0.0.1:34675/status,Memory: 6.05 GiB
Nanny: tcp://127.0.0.1:41319,


## Read the composites as one point layer

`load_composites` reads every CSV and joins them on the lattice coordinates, giving one point
per lattice cell and one column per acquisition named `chlor_<year>_<doy>`. The join is on
rounded coordinates, so grids that differ by floating point noise still line up.

In [4]:
points = oc.load_composites(paths)
print(points.shape)
points.iloc[:3, :6]

(3600, 27)


,longitude,latitude,chlor_2016015,chlor_2016060,chlor_2016135,chlor_2016200
0,-40.0000,25.0,0.2793,0.2764,0.1339,0.0527
1,-39.4915,25.0,0.2606,0.2398,0.1448,0.0625
2,-38.9831,25.0,0.2148,0.2320,0.1654,0.0574


## Bin to hexes and build seasonal covariates

`build_hex_covariates` lays a hex grid over the points (radius three times the lattice
spacing unless given), joins points to hexes with `dask_geopandas.sjoin`, and averages each
acquisition per hex. Acquisitions are then labelled by season from the day of year. The default
`SeasonCalendar` is a two season split, winter for day of year below 104 or from 288 and summer
between; `SeasonCalendar.four_seasons()` gives meteorological seasons. Per hex the stack holds:

- `<season>_mean`: mean over years of that season's mean
- `total_mean`: mean over every acquisition
- `<season>_var`: variance across years of the year over year change in that season
- `<year>_var`: variance across seasons of the year over year change within that year

`missingness_report` on the point table gives the fraction of lattice cells with no value per
acquisition, which is where cloud gaps and swath edges show up. Hex means absorb most of it.

In [5]:
stack = oc.build_hex_covariates(points, use_dask=True)
print("hex cells:", len(stack.cells), " covariates:", len(stack.covariate_columns))
print(stack.covariate_columns)
missing = missingness_report(points.drop(columns="geometry"))
missing[missing.index.str.startswith("chlor_")].sort_values(ascending=False).head(6).round(3)

hex cells: 143  covariates: 8
['summer_mean', 'winter_mean', 'total_mean', 'summer_var', 'winter_var', '2017_var', '2018_var', '2019_var']


chlor_2019250    0.088
chlor_2017200    0.088
chlor_2018320    0.085
chlor_2019320    0.085
chlor_2017015    0.085
chlor_2018200    0.083
dtype: float64

In [6]:
stack.covariates.drop(columns="geometry").describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
hex_id,143.0,71.000,41.425,0.000,35.500,71.000,106.500,142.000
summer_mean,143.0,0.427,0.294,0.098,0.213,0.357,0.625,1.731
winter_mean,143.0,0.952,0.547,0.232,0.503,0.849,1.423,2.731
total_mean,143.0,0.690,0.417,0.166,0.359,0.603,1.036,2.231
summer_var,143.0,0.005,0.009,0.000,0.002,0.003,0.005,0.093
winter_var,143.0,0.004,0.005,0.000,0.002,0.003,0.004,0.044
2017_var,143.0,0.001,0.004,0.000,0.000,0.000,0.001,0.040
2018_var,143.0,0.001,0.009,0.000,0.000,0.000,0.001,0.111
2019_var,143.0,0.001,0.003,0.000,0.000,0.000,0.001,0.024


## Screen for spatial clusters

`screen_hotspots` runs one local statistic per covariate over a distance band weights matrix
(threshold set so every hex has at least one neighbour). Gi* returns a z-score and a
permutation p-value per hex and classes hexes as hot or cold at 90, 95 and 99 percent
confidence. Local Moran's I classes hexes as HH, LL, HL, LH or not significant. Both use 499
permutations here; 999 is the usual choice for a report and takes about twice as long.

In [7]:
gstar = oc.screen_hotspots(
    stack.covariates,
    ["winter_mean", "summer_mean", "summer_var", "total_mean"],
    method="gstar",
    permutations=499,
)
for name, res in gstar.items():
    print(name)
    print(res.summary().to_string(), "\n")

winter_mean
winter_mean_gi_class
not significant    137
hot 95%              4
hot 90%              2 

summer_mean
summer_mean_gi_class
not significant    134
hot 95%              4
hot 99%              4
hot 90%              1 

summer_var
summer_var_gi_class
not significant    141
hot 99%              2 

total_mean
total_mean_gi_class
not significant    135
hot 95%              4
hot 90%              3
hot 99%              1 



In [8]:
lisa = oc.screen_hotspots(stack.covariates, ["2018_var"], method="lisa", permutations=499)
lisa["2018_var"].summary()

2018_var_lisa_class
not significant    128
low-low             12
low-high             2
high-high            1
Name: count, dtype: int64

## Correct for multiple testing

Every hex gets its own test, so at the 5 percent level 5 percent of hexes are flagged by chance
alone. `benjamini_hochberg` adjusts the permutation p-values to control the false discovery rate
and reports how many survive.

In [9]:
layer = gstar["summer_mean"].layer
bh = benjamini_hochberg(layer["summer_mean_gi_p_sim"], alpha=0.05)
print("raw p < 0.05:", int((layer["summer_mean_gi_p_sim"] < 0.05).sum()))
print("BH significant:", int(bh["significant"].sum()), "of", len(bh))

raw p < 0.05: 58
BH significant: 43 of 143


In [10]:
merged = oc.merge_hotspot_layers({**gstar, **lisa})
class_cols = [c for c in merged.columns if c.endswith("_class")]
merged[["hex_id", *class_cols]].head()

,hex_id,winter_mean_gi_class,summer_mean_gi_class,summer_var_gi_class,total_mean_gi_class,2018_var_lisa_class
0,0,not significant,not significant,not significant,not significant,not significant
1,1,not significant,not significant,not significant,not significant,not significant
2,2,not significant,not significant,not significant,not significant,not significant
3,3,not significant,not significant,not significant,not significant,not significant
4,4,not significant,not significant,not significant,not significant,not significant


## Kepler.gl map

The cell below renders the map inline. If the widget shows as blank text, run
`jupyter nbextension enable --py --sys-prefix keplergl` once in the environment and reload the
page; `fortress_gis.viz.kepler.enable_nbextension()` prints the same command. The HTML export in
the last section does not need the extension and opens in any browser.

In [11]:
from fortress_gis.viz.kepler import HOTSPOT_CLASS_COLORS

builder = KeplerMapBuilder(title="Seasonal hotspots", height=550)
builder.add_layer(stack.covariates, "summer mean", color_field="summer_mean", opacity=0.6)
builder.add_layer(
    gstar["summer_mean"].layer,
    "summer mean Gi*",
    color_field="summer_mean_gi_class",
    categorical_colors=HOTSPOT_CLASS_COLORS,
    opacity=0.7,
)
builder.widget() if kepler_available() else print("keplergl not installed")

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


KeplerGl(config={'version': 'v1', 'config': {'visState': {'layers': [{'id': 'summer_mean', 'type': 'geojson', …

## Export

The QGIS bundle holds the hex covariates and one layer per screened attribute with a
categorized QML style matching the Kepler colours. The Kepler HTML has every layer with the
covariate layers hidden by default.

In [12]:
paths_out = oc.export_artifacts(stack, {**gstar, **lisa}, EXPORTS, name="composites")
for k, v in paths_out.items():
    print(f"{k:>10}: {v.relative_to(ROOT)}")
client.close()

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


Map saved to /mnt/c/Users/joogl/OneDrive/Documents/VisualStudioCodeProjects/Fortress-GIS-Projects/exports/oceanography/composites_kepler.html!
      qgis: exports/oceanography/qgis
kepler_html: exports/oceanography/composites_kepler.html
kepler_config: exports/oceanography/composites_kepler_config.json
